# SHIVA-LLM Objective 02 — Stage 5
## Comprehensive simulation-based policy evaluation

**Google Colab:** Python 3, **CPU**, 12–16 GB RAM recommended.

**Required uploads**
- `objective02_stage4a_simulator.zip`
- `objective02_stage4ac_reward_redesign.zip`
- `objective02_stage4b_policy_optimization.zip`

This is a **one-step simulated contextual-bandit evaluation**. Its action effects, challenge credit, costs and reward weights are assumptions, not causal estimates. Stage 4B selected the learned policy using validation data; this notebook **does not refit or reselect it on test data**. The locked test split is used for analysis only.

**Primary comparison:** frozen learned policy vs readiness rule, best fixed policy and random policy in the moderate scenario. Secondary analyses vary action effects and reward weights.

In [ ]:
import os, io, json, zipfile, pathlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import joblib
from google.colab import files
SEED=42
OUT=pathlib.Path('/content/stage5_results');OUT.mkdir(parents=True,exist_ok=True)
names=['objective02_stage4a_simulator.zip','objective02_stage4ac_reward_redesign.zip','objective02_stage4b_policy_optimization.zip']
uploads=files.upload()
assert all(os.path.exists(n) for n in names),f'Missing: {[n for n in names if not os.path.exists(n)]}'
z=[zipfile.ZipFile(n) for n in names]
assert all(x.testzip() is None for x in z),'ZIP integrity failed'
for name,zz in zip(names,z):print(name,zz.namelist())

In [ ]:
p=pd.read_csv(io.BytesIO(z[0].read('response_model_predictions.csv')))
spec=json.loads(z[0].read('simulator_spec.json'))
redesign=json.loads(z[1].read('stage4ac_report.json'))
report=json.loads(z[2].read('stage4b_report.json'))
saved=pd.read_csv(io.BytesIO(z[2].read('locked_test_policy_comparison.csv')))
with warnings.catch_warnings():
    warnings.simplefilter('ignore',UserWarning)
    model=joblib.load(io.BytesIO(z[2].read('stage4b_selected_policy.joblib')))
assert {'split','user_id','order_id','observed_correct','p_observed'}<=set(p.columns)
assert set(p.split.unique())=={'train','val','test'}
assert not p.duplicated(['user_id','order_id']).any()
assert p.p_observed.between(0,1).all()
assert p.observed_correct.isin([0,1]).all()
ids={s:set(p.loc[p.split==s,'user_id']) for s in ['train','val','test']}
assert not(ids['train']&ids['val'] or ids['train']&ids['test'] or ids['val']&ids['test'])
assert report['selected_policy']=='tree_depth_2'
assert report['train_only_fit'] and report['test_not_used_for_selection']
assert report['frozen_stage4ac_reward']==redesign['selected_design']
assert getattr(model,'n_features_in_',None)==1
test=p[p.split=='test'].reset_index(drop=True).copy()
assert len(test)==report['test_events']
print('Test:',len(test),'events;',test.user_id.nunique(),'learners')
print('Frozen model:',report['selected_policy'],'reward:',report['frozen_stage4ac_reward'])

### Reconstruct the frozen simulator

No fitting or selection on the test set. `p_observed` is the pre-response observational correctness prediction. `learned` uses the frozen Stage 4B model. The readiness-rule baseline is reconstructed from the Stage 4B code; the first replication check below will catch a mismatch.

In [ ]:
params=report['reward_parameters']
effects=report['effect_scenarios']
challenge=np.array(report['challenge_credits'],float)
cost=np.array(report['assumed_costs'],float)
ACTIONS=['easy','medium','hard']
p0=test.p_observed.to_numpy()
learner_ids=test.user_id.to_numpy()
def sigmoid(x):return 1/(1+np.exp(-np.clip(x,-35,35)))
def utilities(p,d,g=params):
    pp=np.clip(np.asarray(p),1e-6,1-1e-6)
    logit=np.log(pp/(1-pp))
    q=sigmoid(logit[:,None]+np.array([d,0,-d])[None,:])
    suit=np.exp(-.5*((q-g['tau'])/g['h'])**2)
    u=g['wc']*q+g['wm']*q*challenge[None,:]+g['ws']*suit-g['wk']*cost[None,:]
    return q,u
learned=np.asarray(model.predict(p0.reshape(-1,1)),int)
assert np.isin(learned,[0,1,2]).all()
# Baseline action rules: fixed, seeded random, and predeclared readiness thresholds.
policies={'learned':learned,'fixed_easy':np.zeros(len(test),int),
          'fixed_medium':np.ones(len(test),int),'fixed_hard':np.full(len(test),2,int),
          'readiness_rule':np.where(p0<.4,0,np.where(p0<.7,1,2)).astype(int)}
policies['random']=np.random.default_rng(SEED).integers(0,3,len(test))
print({k:np.bincount(v,minlength=3).tolist() for k,v in policies.items()})

In [ ]:
# Reproduce Stage 4B test comparison before extending it.
def eval_policy(u,q,a):
    n=len(a)
    return {'mean_simulated_reward':float(u[np.arange(n),a].mean()),
            'mean_simulated_regret':float((u.max(axis=1)-u[np.arange(n),a]).mean()),
            'mean_assumed_correctness':float(q[np.arange(n),a].mean())}
rows=[]
for scenario,d in effects.items():
    q,u=utilities(p0,d)
    for name,a in policies.items():
        rows.append({'scenario':scenario,'policy':name,**eval_policy(u,q,a)})
rep=pd.DataFrame(rows)
ref=saved.merge(rep,on=['scenario','policy'],suffixes=('_stage4b','_recomputed'),validate='one_to_one')
diff=(ref.mean_simulated_reward_stage4b-ref.mean_simulated_reward_recomputed).abs()
print('Maximum Stage 4B reward replication difference:',diff.max())
display(ref[['scenario','policy','mean_simulated_reward_stage4b','mean_simulated_reward_recomputed']])
rep.to_csv(OUT/'replication_policy_comparison.csv',index=False)
(OUT/'replication_audit.json').write_text(json.dumps({'max_absolute_reward_difference':float(diff.max()),
  'exact_reproduction_within_1e_8':bool(diff.max()<1e-8),
  'warning':'If reproduction fails, interpret new baseline comparisons as a separate implementation.'},indent=2))

### Paired learner-cluster bootstrap

For each bootstrap draw, resample **learners** with replacement, retaining all their interactions. Compare policies on the **same** sampled contexts. The best-fixed baseline is chosen *within each bootstrap replicate*, which reflects uncertainty in the best-fixed comparator. Confidence intervals quantify variation conditional on simulator assumptions; they do not capture uncertainty in the assumed causal action effects.

In [ ]:
def paired_bootstrap(df,u,acts,scenario,B=500,seed=42):
    rg=np.random.default_rng(seed)
    group_idx=list(df.groupby('user_id').indices.values())
    G=len(group_idx)
    rewards={k:u[np.arange(len(df)),v] for k,v in acts.items()}
    fixed=['fixed_easy','fixed_medium','fixed_hard']
    contrasts=['readiness_rule','random','best_fixed']
    vals={k:[] for k in contrasts}
    for b in range(B):
        draws=rg.integers(0,G,size=G)
        idx=np.concatenate([group_idx[j] for j in draws])
        l=rewards['learned'][idx].mean()
        vals['readiness_rule'].append(l-rewards['readiness_rule'][idx].mean())
        vals['random'].append(l-rewards['random'][idx].mean())
        vals['best_fixed'].append(l-max(rewards[f][idx].mean() for f in fixed))
    result=[]
    for name,x in vals.items():
        x=np.asarray(x)
        result.append({'scenario':scenario,'comparison':'learned_minus_'+name,
            'bootstrap_reps':B,'mean':float(x.mean()),
            'ci_95_low':float(np.quantile(x,.025)),
            'ci_95_high':float(np.quantile(x,.975)),
            'fraction_bootstrap_difference_positive':float((x>0).mean())})
    return result
boots=[]
for i,(scenario,d) in enumerate(effects.items()):
    _,u=utilities(p0,d)
    boots+=paired_bootstrap(test,u,policies,scenario,B=500,seed=SEED+i)
boot=pd.DataFrame(boots)
display(boot)
boot.to_csv(OUT/'paired_learner_bootstrap.csv',index=False)

### Robustness to effect assumptions and reward weights

The frozen policy is never retrained. We vary weights and renormalize the four coefficients. These are sensitivity analyses, **not** fitted preferences. We include a zero-effect scenario as a negative control.

In [ ]:
weight_variants={
 'frozen':params,
 'more_correctness':{**params,'wc':.50,'wm':.20,'ws':.25,'wk':.05},
 'more_challenge':{**params,'wc':.20,'wm':.55,'ws':.20,'wk':.05},
 'more_suitability':{**params,'wc':.20,'wm':.25,'ws':.50,'wk':.05},
 'more_cost':{**params,'wc':.25,'wm':.35,'ws':.20,'wk':.20},
}
rob=[]
for wname,g in weight_variants.items():
    assert abs(sum(g[k] for k in ['wc','wm','ws','wk'])-1)<1e-9
    for scenario,d in effects.items():
        q,u=utilities(p0,d,g)
        vals={k:float(u[np.arange(len(test)),a].mean()) for k,a in policies.items()}
        best_fixed=max(vals[k] for k in ['fixed_easy','fixed_medium','fixed_hard'])
        rob.append({'reward_variant':wname,'scenario':scenario,
            'learned':vals['learned'],'readiness_rule':vals['readiness_rule'],
            'best_fixed':best_fixed,'random':vals['random'],
            'learned_minus_rule':vals['learned']-vals['readiness_rule'],
            'learned_minus_best_fixed':vals['learned']-best_fixed,
            'oracle_minus_learned':float(u.max(axis=1).mean()-vals['learned'])})
rob=pd.DataFrame(rob)
display(rob)
rob.to_csv(OUT/'reward_weight_robustness.csv',index=False)

In [ ]:
# Readiness-stratified policy comparisons and action choices
bins=[0,.4,.7,1.000001]
labels=['low','medium','high']
test['readiness']=pd.cut(test.p_observed,bins=bins,labels=labels,include_lowest=True,right=False)
subrows=[]
for scenario,d in effects.items():
    q,u=utilities(p0,d)
    for label in labels:
        mask=(test.readiness==label).to_numpy()
        if not mask.any():continue
        ix=np.flatnonzero(mask)
        for name,a in policies.items():
            subrows.append({'scenario':scenario,'readiness':label,'policy':name,'events':len(ix),
                'learners':int(test.loc[mask,'user_id'].nunique()),
                'mean_reward':float(u[ix,a[ix]].mean()),
                'fraction_easy':float((a[ix]==0).mean()),
                'fraction_medium':float((a[ix]==1).mean()),
                'fraction_hard':float((a[ix]==2).mean())})
sub=pd.DataFrame(subrows)
sub.to_csv(OUT/'readiness_stratified_comparison.csv',index=False)
display(sub[(sub.scenario=='moderate') & sub.policy.isin(['learned','readiness_rule'])])

In [ ]:
# Multi-seed random baseline: reproducibility / Monte Carlo variance
random_rows=[]
for scenario,d in effects.items():
    _,u=utilities(p0,d)
    for seed in range(30):
        a=np.random.default_rng(seed).integers(0,3,len(test))
        random_rows.append({'scenario':scenario,'seed':seed,
                            'mean_reward':float(u[np.arange(len(test)),a].mean())})
random_df=pd.DataFrame(random_rows)
random_df.to_csv(OUT/'random_baseline_30_seeds.csv',index=False)
display(random_df.groupby('scenario').mean_reward.agg(['mean','std','min','max']))

In [ ]:
# Outcome calibration by readiness: observational response model only.
from sklearn.metrics import roc_auc_score,brier_score_loss,log_loss
cal=[]
for label in labels:
    part=test[test.readiness==label]
    y=part.observed_correct.to_numpy()
    pp=part.p_observed.to_numpy()
    cal.append({'readiness':label,'events':len(part),'learners':part.user_id.nunique(),
                'observed_correctness':float(y.mean()),'predicted_correctness':float(pp.mean()),
                'brier':float(brier_score_loss(y,pp)),
                'auc':float(roc_auc_score(y,pp)) if len(np.unique(y))==2 else None})
cal=pd.DataFrame(cal)
display(cal)
cal.to_csv(OUT/'observational_calibration_by_readiness.csv',index=False)

In [ ]:
# Plots
fig,axs=plt.subplots(1,2,figsize=(12,4))
moderate=rep[rep.scenario=='moderate'].set_index('policy')
order=['learned','readiness_rule','fixed_easy','fixed_medium','fixed_hard','random']
axs[0].bar(range(len(order)),moderate.loc[order,'mean_simulated_reward'])
axs[0].set_xticks(range(len(order)),order,rotation=40,ha='right')
axs[0].set_ylabel('Assumed mean utility');axs[0].set_title('Moderate scenario')
rr=rob[rob.reward_variant=='frozen'].set_index('scenario').reindex(list(effects))
axs[1].bar(range(len(rr)),rr.learned_minus_best_fixed)
axs[1].axhline(0,linewidth=1)
axs[1].set_xticks(range(len(rr)),rr.index)
axs[1].set_ylabel('Learned minus best fixed');axs[1].set_title('Effect sensitivity')
fig.tight_layout()
fig.savefig(OUT/'stage5_policy_comparisons.png',dpi=180)
plt.show()

In [ ]:
# Interpretive guardrails and report
mod=rob[(rob.reward_variant=='frozen')&(rob.scenario=='moderate')].iloc[0]
audit={
 'stage':'5',
 'title':'Comprehensive simulated contextual-bandit evaluation',
 'inputs':names,'frozen_policy':report['selected_policy'],
 'frozen_reward':report['frozen_stage4ac_reward'],
 'moderate_learned_minus_readiness_rule':float(mod.learned_minus_rule),
 'moderate_learned_minus_best_fixed':float(mod.learned_minus_best_fixed),
 'replication_max_absolute_reward_difference':float(diff.max()),
 'stage4b_replication_pass':bool(diff.max()<1e-8),
 'held_out_test_used_only_for_analysis':True,
 'test_events':len(test),'test_learners':int(test.user_id.nunique()),
 'real_causal_effect_identified':False,'real_learning_improvement_established':False,
 'sequential_rl_evaluated':False,
 'caveats':['One-step synthetic actions; no authenticated interventions.',
            'Reward weights, challenge credits, costs, and effects are assumed.',
            'Oracle and learned policy share the same synthetic reward specification.',
            'Bootstrap intervals exclude uncertainty about action-effect assumptions.',
            'Test sensitivity results are exploratory and cannot be used to tune policy without a new holdout.']}
(OUT/'stage5_report.json').write_text(json.dumps(audit,indent=2))
(OUT/'README.txt').write_text('SHIVA-LLM Stage 5: simulated contextual-bandit comparisons only.\n'
 'Run all notebook cells with Stage 4A, 4A-C and 4B ZIP inputs.\n'
 'No observed causal learning effects or sequential RL claims.\n')
archive=pathlib.Path('/content/objective02_stage5_comparative_evaluation.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zz:
    for file in sorted(OUT.iterdir()):
        if file.is_file():zz.write(file,arcname=file.name)
print('Output:',archive,'size:',archive.stat().st_size,'bytes')
files.download(str(archive))